# 01 — Experiment Design

Before running a single line of statistical code, good A/B testing starts with a well-defined experiment. This notebook covers:

1. **Business context** — what question are we answering?
2. **Hypothesis formulation** — null vs. alternative, one-tailed vs. two-tailed
3. **Metric selection** — primary metric, guardrail metrics, and why this matters
4. **Randomization unit** — what gets assigned to control vs. treatment?
5. **Sample size & power** — how many users do we need before we can trust results?
6. **Test duration** — how long should we run before looking?

---

> **Dataset:** Landing page conversion experiment. A company tested a redesigned landing page against the existing one to see if the new design improved sign-up conversion rate.

## 1. Business Context

Every A/B test starts with a business problem, not a statistics problem.

**Scenario:**  
The product team believes the current landing page has friction — users are confused by the layout and leaving before signing up. A designer has built a new version with a clearer value proposition and a more prominent CTA button.

**Business question:**  
> Does the new landing page design increase the rate at which visitors sign up?

**Why this matters:**  
If the conversion rate is currently 12% and we get 50,000 visitors/week, a +2 percentage point lift means ~1,000 additional sign-ups per week. At a $40 average revenue per user, that's $40k/week in incremental revenue. Framing the stakes before the test tells us whether the engineering cost of the experiment is justified.

---

**Key design decisions made upfront:**

| Decision | Choice | Rationale |
|---|---|---|
| What we're changing | Landing page design | Isolated change — no other variables |
| What we're measuring | Sign-up conversion rate | Direct proxy for the business goal |
| Who is eligible | All new visitors | Returning users have prior exposure bias |
| Split | 50/50 control / treatment | Maximizes statistical power for a given total N |

## 2. Hypothesis Formulation

A/B testing is a form of **hypothesis testing**. We define two competing hypotheses before looking at any data.

### Null Hypothesis (H₀)
The new page has no effect on conversion rate. Any observed difference is due to random chance.

$$H_0: p_{\text{treatment}} - p_{\text{control}} = 0$$

### Alternative Hypothesis (H₁)
The new page changes the conversion rate.

$$H_1: p_{\text{treatment}} - p_{\text{control}} \neq 0$$

### One-tailed vs. Two-tailed

We use a **two-tailed test** here. That means we're willing to detect both improvements *and* regressions.

- **Two-tailed:** Did the page *change* conversion? (can go either direction)  
- **One-tailed:** Did the page *increase* conversion? (directional claim)

One-tailed tests have more statistical power (easier to reach significance) but are only appropriate when a regression would never be acted on — which is rarely true in practice. A new page that *hurts* conversion is important to know about.

## 3. Metric Selection

### Primary Metric
The single metric the experiment is designed to move. We succeed or fail based on this.

$$\text{Conversion Rate} = \frac{\text{users who signed up}}{\text{users who visited the page}}$$

A good primary metric is:
- **Sensitive** — it moves if the change has an effect
- **Measurable** — we can compute it cleanly from logs
- **Timely** — it resolves within the test window (not weeks later)
- **Causally linked** — it reflects the outcome we actually care about

### Guardrail Metrics
Metrics we don't expect to move, but monitor to detect unintended harms.

| Guardrail | Why we watch it |
|---|---|
| Page load time | A new design might be heavier and slow the page |
| Bounce rate | A confusing redesign might cause more immediate exits |
| Error rate | New code could introduce form submission failures |

If a guardrail moves negatively, we don't ship — even if the primary metric improved.

### What we're NOT using as primary
- **Revenue per user** — too noisy, needs larger sample, may not resolve in window
- **30-day retention** — doesn't resolve during a typical test duration
- **Click-through rate on the CTA** — intermediate metric; sign-up is the actual goal

## 4. Randomization Unit

The **randomization unit** is what gets assigned to control or treatment. The right choice avoids contamination between groups.

**Our choice: User (by user ID or cookie)**

This means:
- Each visitor is assigned once and sees the same page on every visit
- A user in control never sees the treatment page (and vice versa)

### Why not randomize by session?
If the same user could see both versions across sessions, their behavior in one session could be influenced by the other — this is **carryover contamination**. A user who sees the new page Monday and the old page Tuesday is not an independent observation.

### SUTVA — Stable Unit Treatment Value Assumption
A core assumption of A/B testing: **one user's treatment doesn't affect another user's outcome**. This holds for landing pages (users don't interact with each other during sign-up). It would *not* hold for a social feature like "invite a friend" — there, treating one user changes the experience of their friends in the control group.

## 5. Sample Size & Statistical Power

This is the most commonly skipped step — and skipping it is why most failed A/B tests are uninterpretable.

### Key parameters

| Parameter | Symbol | Our value | What it means |
|---|---|---|---|
| Baseline conversion rate | p₀ | 12% | Current rate from historical data |
| Minimum detectable effect | MDE | +2pp | Smallest lift worth shipping for |
| Significance level | α | 5% | False positive rate we'll tolerate |
| Statistical power | 1−β | 80% | Probability of detecting a real effect |

### What is the MDE?
The **minimum detectable effect** is a business decision, not a statistical one. Ask: "What is the smallest improvement that would justify shipping this change, given its cost?" If a +0.1pp lift doesn't move the needle for the business, don't size the test for it — you'll need millions of users and weeks of runtime.

### What is power?
Power (1−β) is the probability that the test will correctly detect a real effect if one exists. At 80% power, if the true lift is ≥ MDE, there's an 80% chance we'll see a statistically significant result — and a 20% chance we'll miss it (Type II error).

**Running a test without calculating sample size first means you don't know if you have enough data to detect the effect you care about.**

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from ab_stats import sample_size_per_group

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- Experiment parameters ---
BASELINE_RATE = 0.12   # 12% current conversion rate
MDE           = 0.02   # minimum detectable effect: +2 percentage points
ALPHA         = 0.05   # significance level
POWER         = 0.80   # desired power

n = sample_size_per_group(BASELINE_RATE, MDE, ALPHA, POWER)

print(f"Baseline conversion rate : {BASELINE_RATE:.0%}")
print(f"Minimum detectable effect: +{MDE:.0%} ({BASELINE_RATE:.0%} → {BASELINE_RATE+MDE:.0%})")
print(f"Significance level (α)   : {ALPHA}")
print(f"Statistical power (1−β)  : {POWER:.0%}")
print()
print(f"Required sample size: {n:,} users per group")
print(f"Total users needed  : {2*n:,} (both groups combined)")

### How sensitivity changes with MDE

Smaller effects require disproportionately larger samples. This is one of the most important intuitions in A/B testing.

In [ ]:
mde_range = np.arange(0.005, 0.051, 0.001)
sizes = [sample_size_per_group(BASELINE_RATE, d, ALPHA, POWER) for d in mde_range]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(mde_range * 100, [s * 2 for s in sizes], color='steelblue', linewidth=2)

# Annotate our chosen MDE
chosen_total = sample_size_per_group(BASELINE_RATE, MDE, ALPHA, POWER) * 2
ax.axvline(MDE * 100, color='tomato', linestyle='--', linewidth=1.2, label=f'Our MDE: +{MDE:.0%}')
ax.axhline(chosen_total, color='tomato', linestyle=':', linewidth=1)
ax.annotate(f'{chosen_total:,} users', xy=(MDE * 100, chosen_total),
            xytext=(MDE * 100 + 0.5, chosen_total + 15000),
            arrowprops=dict(arrowstyle='->', color='tomato'), color='tomato', fontsize=10)

ax.set_xlabel('Minimum Detectable Effect (percentage points)', fontsize=11)
ax.set_ylabel('Total users required', fontsize=11)
ax.set_title('Sample size grows rapidly as MDE shrinks', fontsize=13)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.show()

### How power and alpha affect required sample size

In [ ]:
power_levels = [0.70, 0.80, 0.90, 0.95]
alpha_levels = [0.01, 0.05, 0.10]

print(f"{'Power':<8} {'α=0.01':>12} {'α=0.05':>12} {'α=0.10':>12}")
print('-' * 48)
for pwr in power_levels:
    row = f"{pwr:.0%}    "
    for a in alpha_levels:
        n_total = sample_size_per_group(BASELINE_RATE, MDE, a, pwr) * 2
        row += f"  {n_total:>10,}"
    print(row)

print()
print("Each cell = total users needed (both groups), MDE = +2pp, baseline = 12%")

**Read this table as a cost matrix.** Demanding higher power or lower α isn't free — it costs users and time. Typical industry standard: α = 0.05, power = 0.80. Raising power to 90% on a tight traffic budget may mean running the test for months.

## 6. Test Duration

Sample size tells us *how many users*, but we also need to translate that to *how many days*.

$$\text{Duration (days)} = \frac{\text{Total users needed}}{\text{Daily eligible visitors}}$$

But there are practical constraints on top of the math:

In [ ]:
daily_visitors = 5_000  # example: 5,000 new visitors per day

total_needed = sample_size_per_group(BASELINE_RATE, MDE, ALPHA, POWER) * 2
raw_days = total_needed / daily_visitors

# Round up to full weeks so we capture complete weekly cycles
weeks = int(np.ceil(raw_days / 7))
duration_days = weeks * 7

print(f"Total users needed    : {total_needed:,}")
print(f"Daily eligible visitors: {daily_visitors:,}")
print(f"Raw days required     : {raw_days:.1f}")
print(f"Rounded to full weeks : {duration_days} days ({weeks} week{'s' if weeks != 1 else ''})")

### Why round to full weeks?

User behavior has weekly seasonality — conversion rates on Monday look different from Saturday. If you run a test for 10 days, you capture 2 full weekends but only 1 of some weekdays. The groups may have been randomized correctly, but the *time distribution* is uneven.

**Always run for at least one full week. Prefer two.**

### Other duration rules

| Rule | Reason |
|---|---|
| Minimum 1–2 weeks regardless of N | Captures weekly seasonality |
| Avoid running over major holidays | Behavioral outliers that don't generalize |
| Don't stop early when you see significance | Inflates false positive rate (peeking problem) |
| Don't extend after seeing non-significance | Inflates false positive rate equally |
| Set end date before launch | Removes experimenter bias from stopping decision |

### The Peeking Problem

If you check results daily and stop the moment p < 0.05, you're running many hypothesis tests — one per day — at the same α level. The probability of a false positive accumulates. With daily checks over 14 days, your true false positive rate can be 3–4× higher than the nominal 5%.

**Fix:** Commit to the end date before launch and look once. If you need interim looks, use sequential testing methods (alpha spending, SPRT) — covered in notebook 06.

## 7. Pre-Experiment Checklist

Before flipping the switch, confirm:

- [ ] **Hypothesis written down** with specific, measurable prediction
- [ ] **Primary metric defined** and instrumentation verified
- [ ] **Guardrail metrics identified** and monitoring set up
- [ ] **Sample size calculated** based on realistic baseline and MDE
- [ ] **Test duration fixed** and end date committed to
- [ ] **Randomization logic reviewed** — no bugs in assignment code
- [ ] **AA test planned** or historical SRM check done (notebook 02)
- [ ] **Stakeholders aligned** on what a "win" vs. "loss" vs. "inconclusive" means

---

## Summary

| Decision | Our choice |
|---|---|
| Business question | Does the new page increase sign-up conversion? |
| Null hypothesis | p_treatment − p_control = 0 |
| Test type | Two-sided z-test for proportions |
| Primary metric | Conversion rate (binary) |
| Randomization unit | User (by cookie/ID) |
| Baseline rate | 12% |
| MDE | +2 percentage points |
| α | 0.05 |
| Power | 80% |
| Sample size | ~3,800 per group (~7,600 total) |
| Duration | 2 weeks (given 5k daily visitors) |

**Next:** [02 — Data Validation](02_data_validation.ipynb) — checking whether the data we collected is trustworthy before analyzing it.